# Chatbot with Memory and Streaming

In [1]:
from typing import Annotated

from typing_extensions import TypedDict

from langgraph.graph import StateGraph,START,END
from langgraph.graph.message import add_messages

In [2]:
class State(TypedDict):
    messages: Annotated[list, add_messages]

graph_builder = StateGraph(State);

In [3]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [4]:
from langchain_groq import ChatGroq

groq_api_key=os.getenv("GROQ_API_KEY")
groq_model_name=os.getenv("GROQ_MODEL_NAME")
llm=ChatGroq(groq_api_key=groq_api_key,model_name=groq_model_name)

In [5]:
# This is node function for Chatbot Node
def chatbot(state: State):
    return {"messages": [llm.invoke(state["messages"])]}

### Memory uses with langgraph

In [6]:
# Now let's add node and edges in graph_builder

# Adding nodes 
graph_builder.add_node("llmchatbot", chatbot)
# Here first parameter("llmchatbot") is the node name and second parameter(chatbot) is the node function. We will use name name in the add_edge function.

# Adding edges
graph_builder.add_edge(START, "llmchatbot")
graph_builder.add_edge("llmchatbot", END)

### Without memory

In [13]:
graph=graph_builder.compile()

In [14]:
# First invoke of the graph
from langchain.messages import HumanMessage


response = graph.invoke(
    {
        "messages": [
            HumanMessage("Who invented Python?")
        ]
    }
)

response["messages"][-1].content

'Python was invented by Guido van Rossum, a Dutch computer programmer. He began working on Python in the late 1980s and released the first version, Python 0.9.1, in 1991. Van Rossum is still involved with the Python community and is often referred to as the "Benevolent Dictator for Life" (BDFL) of the Python project.'

In [15]:
# Second invoke of the graph
response = graph.invoke(
    {
        "messages": [
            HumanMessage("When was it inveted?")
        ]
    },
    config=config
)

response["messages"][-1].content

"It seems like you're asking about the invention of something, but you haven't specified what that is. Could you please provide more details or clarify what you're referring to? I'll do my best to provide the information you're looking for."

##### SO here we can see in the second invoke it is not able to understand about what topic we are asking the question. It is because here every invoke is like a new start without any memory or previous context.

# With memory, same questions

In [8]:
# Creating the memory
from langgraph.checkpoint.memory import MemorySaver

memory = MemorySaver()

In [9]:
# Complile the graph
graph=graph_builder.compile(checkpointer=memory)

#### Memory can store multiples states. So with memory config "thread_id" is used to identify which state the conversation belongs to.

In [10]:
config = {
    "configurable": {
        "thread_id": "chat1"
    }
}

In [11]:
# First invoke of the graph
from langchain.messages import HumanMessage


response = graph.invoke(
    {
        "messages": [
            HumanMessage("Who invented Python?")
        ]
    },
    config=config
)

response["messages"][-1].content

'Python was invented by Guido van Rossum, a Dutch computer programmer. He began working on Python in the late 1980s and released the first version, Python 0.9.1, in 1991. Van Rossum is still involved in the development of Python and is often referred to as the "Benevolent Dictator for Life" (BDFL) of the Python project.'

In [ ]:
# Second invoke of the graph
response = graph.invoke(
    {
        "messages": [
            HumanMessage("When was it inveted?")
        ]
    },
    config=config
)

response["messages"][-1].content

'Python was invented in the late 1980s. The first version, Python 0.9.1, was released in February 1991. Guido van Rossum began working on Python in December 1989, and he released the first version about a year later.'

##### Here we can see in the second invoke it is able to understand about what topic we are talking even without using it name. It is because from memory it got the context from previous conversation.

# Streaming in langgraph

### Stream method

##### Update mode

In [17]:
config = {"configurable": {"thread_id": "3"}}

for chunk in graph.stream({'messages':"Hi,My name is Krish And I like cricket"},config,stream_mode="updates"):
    print(chunk)

{'llmchatbot': {'messages': [AIMessage(content="Hi Krish, nice to meet you. Cricket is an exciting sport, isn't it? Which team or player is your favorite? Are you more into international cricket or do you follow any domestic leagues like the IPL?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 44, 'prompt_tokens': 45, 'total_tokens': 89, 'completion_time': 0.160728371, 'completion_tokens_details': None, 'prompt_time': 0.001359078, 'prompt_tokens_details': None, 'queue_time': 0.163018701, 'total_time': 0.162087449}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_45180df409', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019fc7bf-a26a-7973-ad41-1565d770525c-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 45, 'output_tokens': 44, 'total_tokens': 89})]}}


##### Values mode

In [19]:
for chunk in graph.stream({'messages':"I also like football"},config,stream_mode="values"):
    print(chunk)

{'messages': [HumanMessage(content='I also like football', additional_kwargs={}, response_metadata={}, id='7935a2d5-5648-457f-81fd-d25fb16b7022')]}
{'messages': [HumanMessage(content='I also like football', additional_kwargs={}, response_metadata={}, id='7935a2d5-5648-457f-81fd-d25fb16b7022'), AIMessage(content='Football is an exciting sport with a huge global following. Are you a fan of a particular team or league, such as the NFL, NCAA, or the English Premier League? Or do you enjoy playing the game yourself?', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 45, 'prompt_tokens': 39, 'total_tokens': 84, 'completion_time': 0.152268325, 'completion_tokens_details': None, 'prompt_time': 0.003205106, 'prompt_tokens_details': None, 'queue_time': 0.055229874, 'total_time': 0.155473431}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}

##### So, here we can see in the Values mode it doesn't only gave last stream(which has HumanMessage as well as AIMessage, but it also give first stream also which only has Human Message(First it gave this))

### Astream method

In [26]:
config = {"configurable": {"thread_id": "5"}}

async for event in graph.astream({"messages":["Hi My name is Krish and I like to play cricket"]}, config, stream_mode="updates", version="v2"):
    print(event["data"]["llmchatbot"]["messages"])

[AIMessage(content='Nice to meet you, Krish. Cricket is a fantastic sport. Are you a fan of any particular team or player? Do you play cricket regularly or is it more of a casual hobby for you?', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 41, 'prompt_tokens': 46, 'total_tokens': 87, 'completion_time': 0.096289909, 'completion_tokens_details': None, 'prompt_time': 0.004197028, 'prompt_tokens_details': None, 'queue_time': 0.126332522, 'total_time': 0.100486937}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019fc7d4-1ea8-7ca2-b851-f0e4750c7ed3-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 46, 'output_tokens': 41, 'total_tokens': 87})]


##### Astream_event method

In [31]:
config = {"configurable": {"thread_id": "6"}}

async for event in graph.astream_events({"messages":["Hi My name is Krish and I like to play cricket"]}, config, stream_mode="updates", version="v2"):
    print(event)

{'event': 'on_chain_start', 'data': {'input': {'messages': ['Hi My name is Krish and I like to play cricket']}}, 'name': 'LangGraph', 'tags': [], 'run_id': '019fc7d5-eba9-7bd3-9b4b-fead619ac88f', 'metadata': {'thread_id': '6', 'ls_integration': 'langgraph'}, 'parent_ids': []}
{'event': 'on_chain_start', 'data': {'input': {'messages': [HumanMessage(content='Hi My name is Krish and I like to play cricket', additional_kwargs={}, response_metadata={}, id='58fd92ce-8b98-4a0f-9fb9-9f53488a01a7')]}}, 'name': 'llmchatbot', 'tags': ['graph:step:1'], 'run_id': '019fc7d5-ebac-71f1-af30-e51d4f1aa4a0', 'metadata': {'thread_id': '6', 'ls_integration': 'langgraph', 'langgraph_step': 1, 'langgraph_node': 'llmchatbot', 'langgraph_triggers': ('branch:to:llmchatbot',), 'langgraph_path': ('__pregel_pull', 'llmchatbot'), 'langgraph_checkpoint_ns': 'llmchatbot:739d080a-3015-540c-74ca-a4de215d339b'}, 'parent_ids': ['019fc7d5-eba9-7bd3-9b4b-fead619ac88f']}
{'event': 'on_chat_model_start', 'data': {'input': {'